# PromptForge Phase 1 — Quality Scorer (Colab)

GPU-first training notebook for **PromptForge-Quality**.

**Runtime → Change runtime type → GPU** (T4 / L4 / A100).

This notebook installs the repo package and runs the same code as:
`python scripts/train_quality.py --require-gpu`


## 0. GPU check


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU runtime before training."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


## 1. Get the repo into `/content/promptModel`

**Option A (recommended):** upload/zip the local repo, or clone from GitHub once published.

**Option B:** mount Google Drive if you keep the project there.


In [ ]:
import os
from pathlib import Path

# --- choose one ---
MODE = "upload"  # "upload" | "github" | "drive"

REPO_DIR = Path("/content/promptModel")

if MODE == "github":
    # Replace with your fork/url after you push
    !git clone https://github.com/YOUR_USER/promptModel.git /content/promptModel
elif MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    # Update this path to your Drive copy of the repo
    REPO_DIR = Path("/content/drive/MyDrive/promptModel")
elif MODE == "upload":
    # Upload/unzip the repo first, e.g.:
    # !unzip -q /content/promptModel.zip -d /content
    if not (REPO_DIR / "src" / "promptforge").exists():
        cwd = Path.cwd()
        if (cwd / "src" / "promptforge").exists():
            REPO_DIR = cwd
        else:
            raise FileNotFoundError(
                "Place the repo at /content/promptModel or set MODE=github/drive"
            )

os.chdir(REPO_DIR)
print("Repo:", REPO_DIR.resolve())
print("CWD:", Path.cwd())
print("Has package:", (REPO_DIR / "src" / "promptforge").exists())


## 2. Install package (editable) + deps


In [ ]:
!pip install -q -U pip
!pip install -q -e .
!pip install -q transformers datasets accelerate scikit-learn pandas numpy scipy evaluate huggingface_hub pyyaml

import promptforge
print("promptforge", promptforge.__version__)


## 3. Train quality scorer (GPU + fp16)


In [ ]:
import os
os.environ["PROMPTFORGE_REQUIRE_GPU"] = "1"

from pathlib import Path
from promptforge.config import QualityScorerConfig
from promptforge.training import train_quality_scorer

config = QualityScorerConfig.from_yaml("configs/quality_scorer.yaml")

# Colab-friendly paths
config.dataset_path = "/content/promptforge_dataset.csv"
config.output_dir = "/content/outputs/promptforge-quality"
config.final_model_dir = "/content/outputs/promptforge-quality-model"

# Match Phase-1 experiment (override if you want a quick smoke run)
config.num_examples = 25000
config.num_train_epochs = 3
config.per_device_train_batch_size = 8
config.prefer_gpu = True
config.use_fp16 = True

print(config)

metrics = train_quality_scorer(config, regenerate_dataset=True)
metrics.keys()


## 4. Smoke evaluation


In [ ]:
import json
from promptforge import PromptForge

pf = PromptForge(
    quality_model_path="/content/outputs/promptforge-quality-model",
    prefer_gpu=True,
)

test_prompts = [
    "Make an app.",
    "Build me a website.",
    "Write something about AI.",
    "Make a Python API for beginners.",
    """Build a production-ready REST API using FastAPI and PostgreSQL.
Implement JWT authentication, request validation, structured
error handling and OpenAPI documentation. The API will be used
by a React frontend. Return the complete project structure,
implementation and example requests.""",
]

for prompt in test_prompts:
    result = pf.analyze(prompt)
    print("=" * 80)
    print(prompt.strip()[:120])
    print("quality_score:", result["quality_score"])
    print(json.dumps(result["dimensions"], indent=2))


## 5. (Optional) Download weights / upload to Hugging Face Hub

Download the folder from Colab files UI:
`/content/outputs/promptforge-quality-model`

Or push:


In [ ]:
# from huggingface_hub import login
# login()  # paste token
#
# !python scripts/export_to_hub.py \
#   --model-dir /content/outputs/promptforge-quality-model \
#   --repo-id YOUR_USER/PromptForge-Quality


## 6. Use locally after training

Copy `promptforge-quality-model` into your local repo:

```bash
# local machine
pip install -e .
promptforge analyze "Build me a website" --model outputs/promptforge-quality-model --json
```
